## Installation
Environment: `mamba create -n scvi-env python=3.12`

Torch (v2.5.1): `mamba install pytorch=2.5.1 torchvision=0.20.1 torchaudio=2.5.1 pytorch-cuda=12.1 -c pytorch -c nvidia`

JAX: `mamba install jax jaxlib=0.4.28=cuda12* -c conda-forge`

scVI: `pip install -U scvi-tools`

ipykernel: `mamba install -n scvi-env ipykernel`

matplotlib: `pip install matplotlib`

Scanpy: `pip install scanpy`

## Tutorial
gimVI: https://docs.scvi-tools.org/en/stable/tutorials/notebooks/spatial/gimvi_tutorial.html

In [2]:
!pip install --quiet scvi-colab
from scvi_colab import install

install()

/home/momo/miniforge3/envs/scvi-env/lib/python3.12/site-packages/scvi_colab/_core.py:47: UserWarning: 
                Not currently in Google Colab environment.

                Please run with `run_outside_colab=True` to override.

                Returning with no further action.
                
  warn(


In [3]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())

12.1
True


In [4]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import scvi
import seaborn as sns
from scipy.stats import spearmanr
# from scvi.data import cortex, smfish
from scvi.external import GIMVI

/home/momo/miniforge3/envs/scvi-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

Seed set to 0


Last run with scvi-tools version: 1.2.2.post2


In [6]:
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [7]:
train_size = 0.8

In [8]:
expr_path = '/data/scRNA/ABCA/AIBS/AWS/expression_matrices/MERFISH-C57BL6J-638850/20230830/C57BL6J-638850-raw-wmeta.h5ad'
spatial_data = sc.read_h5ad(expr_path)
spatial_data = spatial_data[spatial_data.obs['brain_section_label'] == 'C57BL6J-638850.38'].copy()
spatial_data

AnnData object with n_obs × n_vars = 120186 × 550
    obs: 'brain_section_label', 'cluster_alias', 'average_correlation_score', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'x', 'y', 'z', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color'
    var: 'gene_symbol', 'transcript_identifier'
    uns: 'accessed_on', 'src'

In [9]:
expr_path = "/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/processed/WMB-10Xv3-Isocortex-1-raw-wmeta.h5ad"
seq_data = sc.read_h5ad(expr_path)
seq_data

AnnData object with n_obs × n_vars = 227670 × 32285
    obs: 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'region_of_interest_order', 'region_of_interest_color'
    var: 'gene_symbol'

In [10]:
# only use genes in both datasets
common_genes = list(set(spatial_data.var_names) & set(seq_data.var_names))
seq_data = seq_data[:, common_genes].copy()

seq_gene_names = seq_data.var_names
n_genes = seq_data.n_vars
n_train_genes = int(n_genes * train_size)

# randomly select training_genes
rand_train_gene_idx = np.random.choice(range(n_genes), n_train_genes, replace=False)
rand_test_gene_idx = sorted(set(range(n_genes)) - set(rand_train_gene_idx))
rand_train_genes = seq_gene_names[rand_train_gene_idx]
rand_test_genes = seq_gene_names[rand_test_gene_idx]

# spatial_data_partial has a subset of the genes to train on
spatial_data_partial = spatial_data[:, rand_train_genes].copy()

# remove cells with no counts
sc.pp.filter_cells(spatial_data_partial, min_counts=1)
sc.pp.filter_cells(seq_data, min_counts=1)

In [18]:
# Normalize and log-transform both datasets
sc.pp.normalize_total(spatial_data, target_sum=1e4)
sc.pp.log1p(spatial_data)

sc.pp.normalize_total(seq_data, target_sum=1e4)
sc.pp.log1p(seq_data)

sc.pp.normalize_total(spatial_data_partial, target_sum=1e4)
sc.pp.log1p(spatial_data_partial)

/home/momo/miniforge3/envs/scvi-env/lib/python3.12/site-packages/scanpy/preprocessing/_normalization.py:208: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [19]:
print(f"Number of spatial_data cells: {spatial_data.n_obs}")
print(f"Number of seq_data cells: {seq_data.n_obs}")

Number of spatial_data cells: 120186
Number of seq_data cells: 227670


In [20]:
print(f"Number of common genes: {len(common_genes)}")

Number of common genes: 500


In [21]:
# setup_anndata for spatial and sequencing data
GIMVI.setup_anndata(spatial_data_partial, labels_key="subclass", batch_key="donor_label")
GIMVI.setup_anndata(seq_data, labels_key="subclass")

# spatial_data should use the same cells as our training data
# cells may have been removed by scanpy.pp.filter_cells()
spatial_data = spatial_data[spatial_data_partial.obs_names]

/home/momo/miniforge3/envs/scvi-env/lib/python3.12/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.X does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/home/momo/miniforge3/envs/scvi-env/lib/python3.12/site-packages/scvi/data/fields/_dataframe_field.py:186: UserWarning: Category 16 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(


In [22]:
# create our model
model = GIMVI(seq_data, spatial_data_partial)

# train for 200 epochs
model.train(200)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/momo/miniforge3/envs/scvi-env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=55` in the `DataLoader` to improve performance.


Epoch 1/200:   0%|          | 0/200 [00:00<?, ?it/s]

/home/momo/miniforge3/envs/scvi-env/lib/python3.12/site-packages/scvi/external/gimvi/_module.py:358: UserWarning: The value argument must be within the support of the distribution
  .log_prob(x)


Epoch 6/200:   2%|▎         | 5/200 [04:42<3:03:24, 56.44s/it, v_num=1]

ValueError: Expected parameter loc (Tensor of shape (128, 10)) of distribution Normal(loc: torch.Size([128, 10]), scale: torch.Size([128, 10])) to satisfy the constraint Real(), but found invalid values:
tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0',
       grad_fn=<AddmmBackward0>)